In [29]:
import os # for handling file paths
import pandas as pd  # data manipulation
import holidays # for handling holidays
from astral.sun import dawn, dusk  # for calculating sunrise/sunset times
from astral import LocationInfo # for defining location for sunrise/sunset calculations

In [30]:
cwd = os.getcwd()

parent_dir = os.path.dirname(cwd)
residential4_data_path = os.path.join(parent_dir, 'data/raw/residential4_merged.csv')
residential4_data = pd.read_csv(residential4_data_path, index_col=0, parse_dates=True)

residential4_data['energy_demand'] = residential4_data['grid_import'] 
residential4_data = residential4_data.drop(columns=['grid_import'])
residential4_data.head(3)

,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,energy_demand
utc_timestamp,,,,,,,,,,,
2015-10-13 17:00:00+00:00,0.002,0.57,0.018,0.0,0.47,0.0,0.002,5.224,0.0,0.0,1.248
2015-10-13 18:00:00+00:00,0.000,0.00,0.026,0.0,0.53,0.0,0.000,4.540,0.0,0.0,0.765
2015-10-13 19:00:00+00:00,0.000,0.00,0.020,0.0,0.43,0.0,0.000,3.843,0.0,0.0,0.701


In [31]:
data = residential4_data[['energy_demand']].copy()

In [32]:
de_holidays = holidays.CountryHoliday(
    'DE',
    subdiv='BW',   # Baden-Württemberg
    years=range(2015, 2018)
)
holiday_dates = pd.to_datetime(list(de_holidays.keys()))
holiday_dates

DatetimeIndex(['2016-01-01', '2016-03-25', '2016-03-28', '2016-05-01',
               '2016-05-05', '2016-05-16', '2016-10-03', '2016-12-25',
               '2016-12-26', '2016-01-06', '2016-05-26', '2016-11-01',
               '2017-01-01', '2017-04-14', '2017-04-17', '2017-05-01',
               '2017-05-25', '2017-06-05', '2017-10-03', '2017-12-25',
               '2017-12-26', '2017-10-31', '2017-01-06', '2017-06-15',
               '2017-11-01', '2015-01-01', '2015-04-03', '2015-04-06',
               '2015-05-01', '2015-05-14', '2015-05-25', '2015-10-03',
               '2015-12-25', '2015-12-26', '2015-01-06', '2015-06-04',
               '2015-11-01'],
              dtype='datetime64[s]', freq=None)

In [33]:
data['is_holiday_or_weekend'] = data.index.to_series().apply(lambda x: x in holiday_dates or x.weekday() >= 5)
data.head(3)

,energy_demand,is_holiday_or_weekend
utc_timestamp,,
2015-10-13 17:00:00+00:00,1.248,False
2015-10-13 18:00:00+00:00,0.765,False
2015-10-13 19:00:00+00:00,0.701,False


In [34]:
city = LocationInfo("Konstanz", "Germany")
data['daylight_flag'] = data.index.to_series().apply(lambda x: dawn(city.observer, date=x.date()) <= x <= dusk(city.observer, date=x.date()))
data.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag
utc_timestamp,,,
2015-10-13 17:00:00+00:00,1.248,False,True
2015-10-13 18:00:00+00:00,0.765,False,False
2015-10-13 19:00:00+00:00,0.701,False,False


In [35]:
def time_of_day(date): # has to be converted to the numerical format to be used in the model
    hour = date.hour
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    else:
        return 'evening'
    
data['time_of_day'] = data.index.to_series().apply(time_of_day)
data.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag,time_of_day
utc_timestamp,,,,
2015-10-13 17:00:00+00:00,1.248,False,True,afternoon
2015-10-13 18:00:00+00:00,0.765,False,False,evening
2015-10-13 19:00:00+00:00,0.701,False,False,evening


In [36]:
def get_season(date):  # has to be converted to the numerical format to be used in the model
    month = date.month
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:
        return 'autumn'
data['season'] = data.index.to_series().apply(get_season)
data.head(3)

,energy_demand,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,
2015-10-13 17:00:00+00:00,1.248,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.765,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.701,False,False,evening,autumn


In [37]:
residential4_data = residential4_data.drop(columns=['energy_demand'])
data2 = data.copy()
residential4_merged_all = residential4_data.join(data2, how='left')
residential4_merged_all.head()

,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,energy_demand,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,,,,,,,,,,,
2015-10-13 17:00:00+00:00,0.002,0.57,0.018,0.0,0.470,0.0,0.002,5.224,0.0,0.0,1.248,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.000,0.00,0.026,0.0,0.530,0.0,0.000,4.540,0.0,0.0,0.765,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.000,0.00,0.020,0.0,0.430,0.0,0.000,3.843,0.0,0.0,0.701,False,False,evening,autumn
2015-10-13 20:00:00+00:00,0.000,0.00,0.017,0.0,0.350,0.0,0.000,3.196,0.0,0.0,0.573,False,False,evening,autumn
2015-10-13 21:00:00+00:00,0.000,0.00,0.029,0.0,0.436,0.0,0.000,2.757,0.0,0.0,0.662,False,False,evening,autumn


In [38]:
cols = ['energy_demand'] + [c for c in residential4_merged_all.columns if c != 'energy_demand']
residential4_merged_all = residential4_merged_all[cols]
residential4_merged_all.head()

,energy_demand,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,,,,,,,,,,,
2015-10-13 17:00:00+00:00,1.248,0.002,0.57,0.018,0.0,0.470,0.0,0.002,5.224,0.0,0.0,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.765,0.000,0.00,0.026,0.0,0.530,0.0,0.000,4.540,0.0,0.0,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.701,0.000,0.00,0.020,0.0,0.430,0.0,0.000,3.843,0.0,0.0,False,False,evening,autumn
2015-10-13 20:00:00+00:00,0.573,0.000,0.00,0.017,0.0,0.350,0.0,0.000,3.196,0.0,0.0,False,False,evening,autumn
2015-10-13 21:00:00+00:00,0.662,0.000,0.00,0.029,0.0,0.436,0.0,0.000,2.757,0.0,0.0,False,False,evening,autumn


In [ ]:
print(residential4_merged_all.shape)
print(residential4_merged_all.isnull().sum())

(12238, 15)
energy_demand                   0
dishwasher                      0
ev                              0
freezer                         0
grid_export                     0
heat_pump                       0
pv                              0
washing_machine                 0
temperature                     0
radiation_direct_horizontal     0
radiation_diffuse_horizontal    0
is_holiday_or_weekend           0
daylight_flag                   0
time_of_day                     0
season                          0
dtype: int64


In [45]:
# save the full processed data
processed_data_dir = os.path.join(parent_dir, 'data/processed')
os.makedirs(processed_data_dir, exist_ok=True)
processed_data_path = os.path.join(processed_data_dir, 'residential4_energy_demand.csv')
residential4_merged_all.to_csv(processed_data_path)

In [46]:
# read newly saved data to verify
verified_data = pd.read_csv(processed_data_path, index_col=0, parse_dates=True)
verified_data.head(3)

,energy_demand,dishwasher,ev,freezer,grid_export,heat_pump,pv,washing_machine,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal,is_holiday_or_weekend,daylight_flag,time_of_day,season
utc_timestamp,,,,,,,,,,,,,,,
2015-10-13 17:00:00+00:00,1.248,0.002,0.57,0.018,0.0,0.47,0.0,0.002,5.224,0.0,0.0,False,True,afternoon,autumn
2015-10-13 18:00:00+00:00,0.765,0.000,0.00,0.026,0.0,0.53,0.0,0.000,4.540,0.0,0.0,False,False,evening,autumn
2015-10-13 19:00:00+00:00,0.701,0.000,0.00,0.020,0.0,0.43,0.0,0.000,3.843,0.0,0.0,False,False,evening,autumn
